# Multilingual Audio Processing with Language Detection and Translation

This notebook processes audio files containing multiple languages (Telugu, Hindi, English), detects each word's language, and translates them to a target language.

## Import Required Libraries

In [ ]:
import os
import whisper
from pydub import AudioSegment
from pydub.silence import split_on_silence
from deep_translator import GoogleTranslator
from langdetect import detect

# Options: tiny, base, small, medium, large
whisper_model = whisper.load_model("small")
print("Whisper model loaded successfully!")

Loading Whisper model...
Whisper model loaded successfully!


In [29]:
# Verify FFmpeg is accessible from system PATH
import subprocess
import shutil

# Check if ffmpeg and ffprobe are in PATH
ffmpeg_path = shutil.which("ffmpeg")
ffprobe_path = shutil.which("ffprobe")

print("FFmpeg found in PATH:", ffmpeg_path if ffmpeg_path else "Not found")
print("FFprobe found in PATH:", ffprobe_path if ffprobe_path else "Not found")

# Test FFmpeg functionality
if ffmpeg_path:
    try:
        result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
        print("\nFFmpeg is working correctly!")
        print("Version:", result.stdout.split('\n')[0])
    except Exception as e:
        print(f"\nError testing FFmpeg: {e}")
else:
    print("\nWarning: FFmpeg not found in PATH. Please restart your terminal/notebook.")

FFmpeg found in PATH: C:\Program Files\ffmpeg\bin\ffmpeg.EXE
FFprobe found in PATH: C:\Program Files\ffmpeg\bin\ffprobe.EXE

FFmpeg is working correctly!
Version: ffmpeg version N-121490-g7e8ef2ded2-20251024 Copyright (c) 2000-2025 the FFmpeg developers


## Function 1: Split Audio into Segments

This function splits the audio file into individual segments based on silence detection.

In [30]:
def audio_to_segments(audio_path):
    """Split audio file into segments based on silence"""
    audio = AudioSegment.from_file(audio_path)
    
    # Split audio on silence (adjust parameters as needed)
    segments = split_on_silence(
        audio,
        min_silence_len=500,  # minimum silence length in ms
        silence_thresh=audio.dBFS - 14,  # silence threshold
        keep_silence=250  # keep some silence at edges
    )
    
    return segments

## Function 2: Detect Language from Text

This function detects the language of transcribed text.

In [31]:
def detect_language_from_text(text):
    """Detect language from text using script analysis and langdetect with smart mapping"""
    try:
        # First check if it contains Devanagari script (Hindi)
        devanagari_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
        if devanagari_chars > 0:
            return 'hindi'
        
        # Check if it contains Telugu script
        telugu_chars = sum(1 for c in text if '\u0C00' <= c <= '\u0C7F')
        if telugu_chars > 0:
            return 'telugu'
        
        # For romanized text, use langdetect with smart mapping
        # langdetect often misidentifies romanized Indian languages
        lang = detect(text)
        
        # Map common false positives for romanized Indian languages
        lang_map = {
            'en': 'english',
            'hi': 'hindi',
            'te': 'telugu',
            'ta': 'telugu',
            'ro': 'hindi',  # Romanian is often misdetected Hindi romanized
            'it': 'telugu',  # Italian is often misdetected Telugu romanized
            'id': 'telugu',  # Indonesian is often misdetected Telugu romanized
            'pt': 'hindi',  # Portuguese sometimes misdetected as Hindi romanized
        }
        
        return lang_map.get(lang, 'english')  # Default to English
    except:
        return 'english'

## Function 3: Transcribe Audio Segment

This function transcribes a single audio segment in multiple languages.

In [50]:
def transcribe_audio_segment(audio_segment):
    """Transcribe a single audio segment using Whisper AI"""
    import time
    
    # Preprocess audio for better recognition
    # Normalize audio volume
    audio_segment = audio_segment.normalize()
    
    # Export segment to temporary file
    temp_file = f"temp_segment_{time.time()}.wav"
    audio_segment.export(
        temp_file, 
        format="wav",
        parameters=["-ar", "16000", "-ac", "1"]  # 16kHz sample rate, mono
    )
    
    try:
        # First detect the language
        audio = whisper.load_audio(temp_file)
        audio = whisper.pad_or_trim(audio)
        mel = whisper.log_mel_spectrogram(audio).to(whisper_model.device)
        _, probs = whisper_model.detect_language(mel)
        detected_lang_code = max(probs, key=probs.get)
        
        # Map language codes to full names
        lang_names = {
            'en': 'English',
            'hi': 'Hindi',
            'te': 'Telugu',
            'ta': 'Tamil',
            'ur': 'Urdu'
        }
        detected_lang_name = lang_names.get(detected_lang_code, detected_lang_code.upper())
        
        # Transcribe in the detected native language
        native_result = whisper_model.transcribe(
            temp_file,
            language=detected_lang_code,
            task='transcribe',
            fp16=False
        )
        native_text = native_result['text'].strip()
        
        # Transcribe in English for clarity
        english_result = whisper_model.transcribe(
            temp_file,
            language='en',
            task='transcribe',
            fp16=False
        )
        english_text = english_result['text'].strip()
        
        if native_text or english_text:
            print(f"    Language: {detected_lang_name}")
            print(f"    Native: {native_text}")
            print(f"    English: {english_text}")
            return native_text, english_text, detected_lang_name
        else:
            return None, None, None
                    
    except Exception as e:
        print(f"    Transcription error: {str(e)[:100]}")
        return None, None, None
                
    finally:
        # Clean up temp file
        try:
            time.sleep(0.1)
            if os.path.exists(temp_file):
                os.remove(temp_file)
        except:
            pass

## Function 4: Translate Text

This function translates text from source language to target language.

In [33]:
def translate_text(text, source_lang, target_lang):
    """Translate text from source to target language"""
    lang_codes = {
        'english': 'en',
        'hindi': 'hi',
        'telugu': 'te'
    }
    
    try:
        src_code = lang_codes.get(source_lang, 'auto')
        dest_code = lang_codes.get(target_lang, 'en')
        
        translation = GoogleTranslator(source=src_code, target=dest_code).translate(text)
        return translation
    except Exception as e:
        print(f"Translation error: {e}")
        return text

## Function 5: Main Processing Function

This is the main function that orchestrates the entire process.

In [51]:
def process_multilingual_audio(audio_path, target_language='english'):
    """Main function to process multilingual audio"""
    print(f"Processing audio file: {audio_path}")
    print(f"Target language: {target_language}\n")
    
    # Split audio into segments
    segments = audio_to_segments(audio_path)
    print(f"Found {len(segments)} segments\n")
    
    results = []
    
    for i, segment in enumerate(segments):
        print(f"Processing segment {i+1}/{len(segments)}...")
        
        # Transcribe segment and detect language
        native_text, english_text, detected_language = transcribe_audio_segment(segment)
        
        if not native_text and not english_text:
            print(f"  No speech detected in segment {i+1}\n")
            continue
        
        results.append({
            'segment': i+1,
            'native_text': native_text,
            'english_text': english_text,
            'language': detected_language
        })
        print()
    
    return results

## Example Usage

Run this cell to process your audio file. Make sure to replace the path with your actual audio file.

In [ ]:
# Example usage
audio_file = "1767323584571329487nzrf5wt8-voicemaker.in-speech.mp3" 

results = process_multilingual_audio(audio_file, target_language='english')

# Display results
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50 + "\n")
for result in results:
    print(f"Segment {result['segment']} [{result['language']}]:")
    print(f"  English: {result['english_text']}")
    print()

# Create complete transcript 
print("\n" + "="*50)
print("COMPLETE TRANSCRIPT (English)")
print("="*50 + "\n")
complete_transcript = " ".join([result['english_text'] for result in results])
print(complete_transcript)

Processing audio file: 1767323584571329487nzrf5wt8-voicemaker.in-speech.mp3
Target language: english

Found 3 segments

Processing segment 1/3...
    Language: FR
    Native: My name is Victor.
    English: My name is Victor.

Processing segment 2/3...
    Language: Telugu
    Native: philanthropy
    English: I am studying at college.

Processing segment 3/3...
    Language: Hindi
    Native: मेरी सब से अच्छी दोस्त अरने है
    English: My best friend is Aranya.


FINAL RESULTS

Segment 1 [FR]:
  English: My name is Victor.

Segment 2 [Telugu]:
  English: I am studying at college.

Segment 3 [Hindi]:
  English: My best friend is Aranya.


COMPLETE TRANSCRIPT (English)

My name is Victor. I am studying at college. My best friend is Aranya.


## Concatenated Transcript

This cell concatenates all segments and displays the complete transcript.

In [ ]:
# Display complete transcript
print("\n" + "="*50)
print("COMPLETE TRANSCRIPT (English)")
print("="*50 + "\n")

complete_transcript_english = " ".join([result['english_text'] for result in results])
print(complete_transcript_english)

print(complete_transcript_native)

print("\n" + "="*50)complete_transcript_native = " ".join([result['native_text'] for result in results])

print("COMPLETE TRANSCRIPT (Native Languages)")
print("="*50 + "\n")


COMPLETE TRANSCRIPT

My name is Victor. I am studying at college. My best friend is Aranya.


## Installation Instructions

Before running this notebook, you need to install the required packages. Run the following command in your terminal:

```bash
pip install openai-whisper pydub deep-translator langdetect numpy
```

You'll also need to install **FFmpeg** for audio processing:
- **Windows**: Download from https://ffmpeg.org/ and add to PATH
- **Linux**: `sudo apt-get install ffmpeg`
- **Mac**: `brew install ffmpeg`

**Note**: This notebook uses Whisper AI, a free and open-source model from OpenAI that provides highly accurate multilingual speech recognition. The first time you run it, Whisper will download the model file (approximately 140 MB for the base model).